# Unidad Modular III · Modelos de primer y segundo orden en Vensim y Python
## Ecuaciones diferenciales, análisis de escenarios y seis casos de ingeniería informática

Este cuaderno acompaña la presentación y el laboratorio web. Hace tres cosas:

1. **Apoya a Vensim.** Crea los seis modelos `.mdl` de la clase y los ejecuta desde Python con **PySD** (o, si PySD no está disponible, con un intérprete mínimo incluido aquí). Así se comparan los resultados de Vensim y Python.
2. **Modela en Python** cada caso con Euler, RK4 y `solve_ivp`, y hace el **análisis de escenarios**: escenarios, políticas, tornado y Monte Carlo.
3. **Genera la plantilla de Excel** `Plantilla_Modelos_Primer_Segundo_Orden.xlsx`, con una hoja por caso, fórmulas de Euler, escenarios y tablero.

| Caso | Orden | Archivo Vensim | Hoja de Excel |
|---|---|---|---|
| 1. Temperatura de la CPU | 1 · lineal | `01_CPU_primer_orden.mdl` | `CPU` |
| 2. Cola de tickets de soporte | 1 · lineal | `02_Cola_tickets_primer_orden.mdl` | `Tickets` |
| 3. Gusano Code-Red (2001) | 1 · no lineal | `03_Gusano_CodeRed_logistico.mdl` | `Gusano` |
| 4. Autoescalado en la nube | 2 y superior | `04_Autoescalado_segundo_orden.mdl` | `Autoescalado` |
| 5. Proyecto y ley de Brooks | 3 | `05_Proyecto_Ley_de_Brooks.mdl` | `Brooks` |
| 6. Ventilador con control PI | 2 · lazo cerrado | `06_Ventilador_PI_segundo_orden.mdl` | `Ventilador` |

**Cómo usarlo:** *Entorno de ejecución → Ejecutar todas*. Al final se descargan la plantilla de Excel y un `.zip` con los seis modelos de Vensim.

In [ ]:
# Instalación de PySD (solo la primera vez en Colab). Si falla, el cuaderno usa su intérprete propio.
try:
    import pysd
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pysd"], check=False)
    try:
        import pysd
    except ImportError:
        pysd = None
print("PySD disponible:", pysd is not None, "· versión", getattr(pysd, "__version__", "—"))

In [ ]:
import math, re, os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
plt.rcParams.update({"figure.figsize": (10, 4.2), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
COL = {"navy": "#0F172A", "cyan": "#0891B2", "amber": "#F59E0B", "rose": "#E11D48", "green": "#16A34A", "indigo": "#6366F1", "slate": "#64748B"}

def euler(f, x0, T, dt, nombres=None):
    """Integra dx/dt = f(t, x) con Euler (el método por defecto de Vensim)."""
    n = int(round(T / dt)); t = np.arange(n + 1) * dt; X = np.zeros((n + 1, len(x0))); X[0] = x0
    for k in range(n): X[k + 1] = X[k] + dt * np.asarray(f(t[k], X[k]), float)
    return pd.DataFrame(X, index=pd.Index(t, name="t"), columns=nombres or [f"x{i}" for i in range(len(x0))])

def rk4(f, x0, T, dt, nombres=None):
    """Runge-Kutta de cuarto orden (opción RK4 de Vensim)."""
    n = int(round(T / dt)); t = np.arange(n + 1) * dt; X = np.zeros((n + 1, len(x0))); X[0] = x0
    for k in range(n):
        x = X[k]; k1 = np.asarray(f(t[k], x), float); k2 = np.asarray(f(t[k] + dt / 2, x + dt / 2 * k1), float); k3 = np.asarray(f(t[k] + dt / 2, x + dt / 2 * k2), float); k4 = np.asarray(f(t[k] + dt, x + dt * k3), float)
        X[k + 1] = x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return pd.DataFrame(X, index=pd.Index(t, name="t"), columns=nombres or [f"x{i}" for i in range(len(x0))])
print("Listo")

---
## 1. Modelos de primer orden
Forma estándar: $\tau\,\dfrac{dx}{dt} + x = x^*$, con solución $x(t) = x^* + (x_0 - x^*)\,e^{-t/\tau}$.

Comparamos la solución exacta con Euler (varios pasos DT), RK4 y `solve_ivp`. Observe que con $DT = \tau$ Euler llega a cero en un paso y con $DT = 1{,}5\tau$ oscila: el error es numérico, no del modelo.

In [ ]:
tau = 10.0; f = lambda t, x: [-x[0] / tau]
t_fino = np.linspace(0, 40, 401)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
ax[0].plot(t_fino, np.exp(-t_fino / tau), color=COL["navy"], lw=2.5, label="Exacta")
for dt, c in [(1, COL["cyan"]), (5, COL["amber"]), (10, COL["rose"]), (15, COL["indigo"])]:
    d = euler(f, [1.0], 40 if dt != 15 else 45, dt); ax[0].plot(d.index, d.x0, "o-", color=c, lw=1.6, ms=4, label=f"Euler DT = {dt}")
ax[0].set(title="Euler con distintos pasos (τ = 10)", xlabel="tiempo", ylabel="x"); ax[0].legend()
err = []
for dt in [0.5, 1, 2, 5]:
    e1 = abs(euler(f, [1.0], 20, dt).x0.iloc[-1] - np.exp(-2)); e2 = abs(rk4(f, [1.0], 20, dt).x0.iloc[-1] - np.exp(-2)); err.append((dt, e1, e2))
err = pd.DataFrame(err, columns=["DT", "error Euler", "error RK4"]).set_index("DT")
ax[1].loglog(err.index, err["error Euler"], "o-", color=COL["rose"], label="Euler (orden 1)"); ax[1].loglog(err.index, err["error RK4"], "s-", color=COL["cyan"], label="RK4 (orden 4)")
ax[1].set(title="Error en t = 20 según el paso", xlabel="DT", ylabel="error absoluto"); ax[1].legend(); plt.tight_layout(); plt.show()
sol = solve_ivp(lambda t, x: -x / tau, (0, 20), [1.0], rtol=1e-9, atol=1e-12)
print("solve_ivp (paso adaptativo) en t = 20:", sol.y[0, -1], " exacto:", np.exp(-2))
err.style.format("{:.2e}")

### Regla del 63 % y línea de fase
En $t = \tau$ se recorre el 63,2 % de la brecha; en $4\tau$, el 98,2 %. En un modelo no lineal $dx/dt = f(x)$ los equilibrios son las raíces de $f$ y son estables si $f'(x^*) < 0$.

In [ ]:
for k in range(1, 6): print(f"{k} τ → {100 * (1 - np.exp(-k)):.1f} % de la brecha")
beta, N = 1.33, 360000
I = np.linspace(0, N, 400); fI = beta * I * (1 - I / N)
plt.plot(I / 1000, fI / 1000, color=COL["indigo"], lw=2.5); plt.axhline(0, color="k", lw=0.8)
plt.scatter([0], [0], s=90, facecolors="white", edgecolors=COL["rose"], lw=2, zorder=5, label="I = 0 (inestable: f'(0) = β > 0)")
plt.scatter([N / 1000], [0], s=90, color=COL["cyan"], zorder=5, label="I = N (estable: f'(N) = −β < 0)")
plt.title("Línea de fase del gusano: dI/dt = βI(1 − I/N)"); plt.xlabel("infectados (miles)"); plt.ylabel("dI/dt (miles por hora)"); plt.legend(); plt.show()

---
## 2. Modelos de segundo orden
Ecuación canónica: $\ddot x + 2\zeta\omega_n\dot x + \omega_n^2 x = \omega_n^2 u$, que se escribe como **dos niveles**: $\dot x_2 = x_1$, $\dot x_1 = \omega_n^2(u - x_2) - 2\zeta\omega_n x_1$.

Métricas (0 < ζ < 1): $M_p = e^{-\pi\zeta/\sqrt{1-\zeta^2}}$, $t_p = \pi/(\omega_n\sqrt{1-\zeta^2})$, $t_s \approx 4/(\zeta\omega_n)$.

In [ ]:
def segundo_orden(zeta, wn=1.0, u=1.0, T=20, dt=0.01):
    return rk4(lambda t, x: [wn ** 2 * (u - x[1]) - 2 * zeta * wn * x[0], x[0]], [0.0, 0.0], T, dt, ["velocidad", "x"])
def metricas(zeta, wn):
    if zeta >= 1: return dict(Mp=0.0, tp=np.nan, ts=4 / (zeta * wn))
    wd = wn * np.sqrt(1 - zeta ** 2); return dict(Mp=np.exp(-np.pi * zeta / np.sqrt(1 - zeta ** 2)), tp=np.pi / wd, ts=4 / (zeta * wn))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4)); filas = []
for z, c in zip([0.1, 0.3, 0.5, 0.7, 1.0, 2.0], [COL["rose"], COL["amber"], COL["indigo"], COL["cyan"], COL["green"], COL["slate"]]):
    d = segundo_orden(z); ax[0].plot(d.index, d.x, color=c, lw=2, label=f"ζ = {z}")
    m = metricas(z, 1.0); filas.append([z, 100 * m["Mp"], 100 * (d.x.max() - 1), m["tp"], np.roots([1, 2 * z, 1])])
ax[0].axhline(1, ls="--", color="gray"); ax[0].set(title="Respuesta al escalón (ωn = 1)", xlabel="tiempo"); ax[0].legend(ncol=2)
for z, c in [(0.3, COL["amber"]), (1.0, COL["green"]), (2.0, COL["slate"]), (-0.2, COL["rose"])]:
    r = np.roots([1, 2 * z, 1]); ax[1].scatter(r.real, r.imag, s=120, marker="x", color=c, lw=3, label=f"ζ = {z}")
ax[1].axvline(0, color="k", lw=1); ax[1].axhline(0, color="k", lw=0.8); ax[1].axvspan(-4, 0, color="#DCFCE7", alpha=0.6)
ax[1].set(xlim=(-4, 1), ylim=(-1.5, 1.5), title="Polos en el plano s (verde = estable)", xlabel="Re(s)", ylabel="Im(s)"); ax[1].legend(); plt.tight_layout(); plt.show()
pd.DataFrame(filas, columns=["ζ", "Mp teórico (%)", "Mp simulado (%)", "tp teórico", "raíces"]).round(2)

---
## 3. Vensim desde Python
La siguiente celda **escribe los seis archivos `.mdl`** en la carpeta `vensim/`. Puede descargarlos y abrirlos en Vensim PLE (gratuito para uso educativo), o ejecutarlos aquí mismo con PySD.

> Los diagramas de los `.mdl` se generaron por programa: al abrirlos en Vensim puede que convenga reacomodar algunas flechas. Las ecuaciones son las mismas que usa este cuaderno.

In [ ]:
MDL = {
    '01_CPU_primer_orden': '{UTF-8}\nTemperatura CPU=\n\tINTEG (\n\t\t\tCalentamiento-Enfriamiento,\n\t\t\t\tTemperatura ambiente+Potencia en reposo*Resistencia termica)\n\t~\tC\n\t~\tNivel: temperatura del procesador. Arranca en el equilibrio de reposo.\n\t|\n\nCalentamiento=\n\tPotencia/Capacidad termica\n\t~\tC/s\n\t~\tFlujo de entrada: la potencia disipada calienta el chip.\n\t|\n\nEnfriamiento=\n\t(Temperatura CPU-Temperatura ambiente)/(Resistencia termica*Capacidad termica)\n\t~\tC/s\n\t~\tFlujo de salida: ley de enfriamiento de Newton.\n\t|\n\nPotencia=\n\tPotencia en reposo+STEP(Potencia en carga-Potencia en reposo, Inicio de la carga)\n\t~\tW\n\t~\tEscalón de carga (por ejemplo, un renderizado).\n\t|\n\nPotencia en reposo=\n\t15\n\t~\tW\n\t~\t\n\t|\n\nPotencia en carga=\n\t95\n\t~\tW\n\t~\t\n\t|\n\nInicio de la carga=\n\t30\n\t~\ts\n\t~\t\n\t|\n\nResistencia termica=\n\t0.4\n\t~\tC/W\n\t~\tDisipador. Pruebe 0.75 (disipador sucio) o 0.25 (refrigeración líquida).\n\t|\n\nCapacidad termica=\n\t50\n\t~\tJ/C\n\t~\tMasa térmica del chip y el disipador.\n\t|\n\nTemperatura ambiente=\n\t25\n\t~\tC\n\t~\t\n\t|\n\nConstante de tiempo=\n\tResistencia termica*Capacidad termica\n\t~\ts\n\t~\ttau = R·C. El 63 % del cambio ocurre en tau segundos.\n\t|\n\nTemperatura de equilibrio=\n\tTemperatura ambiente+Potencia*Resistencia termica\n\t~\tC\n\t~\tHacia donde se dirige la temperatura (objetivo del bucle B).\n\t|\n\n********************************************************\n\t.Control\n********************************************************~\n\t\tSimulation Control Parameters\n\t|\n\nFINAL TIME  = 200\n\t~\ts\n\t~\tTiempo final de la simulación.\n\t|\n\nINITIAL TIME  = 0\n\t~\ts\n\t~\tTiempo inicial de la simulación.\n\t|\n\nSAVEPER  = \n\tTIME STEP\n\t~\ts [0,?]\n\t~\tFrecuencia de guardado.\n\t|\n\nTIME STEP  = 0.5\n\t~\ts [0,?]\n\t~\tPaso de integración (método de Euler).\n\t|\n\n\\\\\\---/// Sketch information - do not modify anything except names\nV300  Do not put anything below this section - it will be ignored\n*View 1\n$192-192-192,0,Times New Roman|12||0-0-0|0-0-0|0-0-255|-1--1--1|-1--1--1|96,96,100,0\n12,1,48,150,260,10,8,0,3,0,0,-1,0,0,0\n10,2,Temperatura CPU,360,260,46,26,3,3,0,0,0,0,0,0\n12,3,48,570,260,10,8,0,3,0,0,-1,0,0,0\n1,4,6,2,4,0,0,22,0,0,0,-1--1--1,,1|(289,260)|\n1,5,6,1,100,0,0,22,0,0,0,-1--1--1,,1|(182,260)|\n11,6,48,248,260,6,8,34,3,0,0,1,0,0,0\n10,7,Calentamiento,248,282,44,12,40,3,0,0,-1,0,0,0\n1,8,10,3,4,0,0,22,0,0,0,-1--1--1,,1|(512,260)|\n1,9,10,2,100,0,0,22,0,0,0,-1--1--1,,1|(405,260)|\n11,10,48,471,260,6,8,34,3,0,0,1,0,0,0\n10,11,Enfriamiento,471,282,44,12,40,3,0,0,-1,0,0,0\n10,12,Potencia,250,150,40,12,8,3,0,0,0,0,0,0\n10,13,Potencia en reposo,120,90,40,12,8,3,0,0,0,0,0,0\n10,14,Potencia en carga,260,70,40,12,8,3,0,0,0,0,0,0\n10,15,Inicio de la carga,380,110,40,12,8,3,0,0,0,0,0,0\n10,16,Capacidad termica,360,380,40,12,8,3,0,0,0,0,0,0\n10,17,Resistencia termica,560,380,40,12,8,3,0,0,0,0,0,0\n10,18,Temperatura ambiente,620,150,40,12,8,3,0,0,0,0,0,0\n10,19,Constante de tiempo,460,450,40,12,8,3,0,0,0,0,0,0\n10,20,Temperatura de equilibrio,200,420,40,12,8,3,0,0,0,0,0,0\n1,21,13,12,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,22,14,12,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,23,15,12,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,24,12,6,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,25,16,6,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,26,2,10,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,27,18,10,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,28,17,10,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,29,16,10,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,30,17,19,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,31,16,19,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,32,12,20,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,33,17,20,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n///---\\\\\\\n:L<%^E!@\n1:Current.vdf\n9:Current\n15:0,0,0,0,0,0\n19:100,0\n27:0,\n34:0,\n4:Time\n5:Temperatura CPU\n35:Date\n36:YYYY-MM-DD\n37:2000\n38:1\n39:1\n40:0\n41:0\n42:1\n24:0\n25:200\n26:200\n',
    '02_Cola_tickets_primer_orden': '{UTF-8}\nTickets pendientes=\n\tINTEG (\n\t\t\tLlegadas-Resoluciones,\n\t\t\t\tLlegadas normales*Tiempo de resolucion)\n\t~\ttickets\n\t~\tNivel: solicitudes sin resolver en la mesa de ayuda. Arranca en el equilibrio de la ley de Little.\n\t|\n\nLlegadas=\n\tLlegadas normales+STEP(Llegadas en incidente-Llegadas normales, Hora del incidente)\n\t~\ttickets/hora\n\t~\t\n\t|\n\nResoluciones=\n\tTickets pendientes/Tiempo de resolucion\n\t~\ttickets/hora\n\t~\tSalida proporcional al nivel: primer orden.\n\t|\n\nLlegadas normales=\n\t40\n\t~\ttickets/hora\n\t~\t\n\t|\n\nLlegadas en incidente=\n\t70\n\t~\ttickets/hora\n\t~\tEscenarios: optimista 55, pesimista 100.\n\t|\n\nHora del incidente=\n\t8\n\t~\thora\n\t~\t\n\t|\n\nTiempo de resolucion=\n\t2\n\t~\thora\n\t~\tCon más agentes baja a 1.4.\n\t|\n\nTiempo de espera estimado=\n\tTickets pendientes/Llegadas\n\t~\thora\n\t~\tLey de Little: W = L / lambda.\n\t|\n\n********************************************************\n\t.Control\n********************************************************~\n\t\tSimulation Control Parameters\n\t|\n\nFINAL TIME  = 24\n\t~\thora\n\t~\tTiempo final de la simulación.\n\t|\n\nINITIAL TIME  = 0\n\t~\thora\n\t~\tTiempo inicial de la simulación.\n\t|\n\nSAVEPER  = \n\tTIME STEP\n\t~\thora [0,?]\n\t~\tFrecuencia de guardado.\n\t|\n\nTIME STEP  = 0.0625\n\t~\thora [0,?]\n\t~\tPaso de integración (método de Euler).\n\t|\n\n\\\\\\---/// Sketch information - do not modify anything except names\nV300  Do not put anything below this section - it will be ignored\n*View 1\n$192-192-192,0,Times New Roman|12||0-0-0|0-0-0|0-0-255|-1--1--1|-1--1--1|96,96,100,0\n12,1,48,150,250,10,8,0,3,0,0,-1,0,0,0\n10,2,Tickets pendientes,360,250,46,26,3,3,0,0,0,0,0,0\n12,3,48,570,250,10,8,0,3,0,0,-1,0,0,0\n1,4,6,2,4,0,0,22,0,0,0,-1--1--1,,1|(289,250)|\n1,5,6,1,100,0,0,22,0,0,0,-1--1--1,,1|(182,250)|\n11,6,48,248,250,6,8,34,3,0,0,1,0,0,0\n10,7,Llegadas,248,272,44,12,40,3,0,0,-1,0,0,0\n1,8,10,3,4,0,0,22,0,0,0,-1--1--1,,1|(512,250)|\n1,9,10,2,100,0,0,22,0,0,0,-1--1--1,,1|(405,250)|\n11,10,48,471,250,6,8,34,3,0,0,1,0,0,0\n10,11,Resoluciones,471,272,44,12,40,3,0,0,-1,0,0,0\n10,12,Llegadas normales,150,130,40,12,8,3,0,0,0,0,0,0\n10,13,Llegadas en incidente,260,100,40,12,8,3,0,0,0,0,0,0\n10,14,Hora del incidente,360,130,40,12,8,3,0,0,0,0,0,0\n10,15,Tiempo de resolucion,560,370,40,12,8,3,0,0,0,0,0,0\n10,16,Tiempo de espera estimado,360,400,40,12,8,3,0,0,0,0,0,0\n1,17,12,6,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,18,13,6,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,19,14,6,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,20,2,10,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,21,15,10,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,22,2,16,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,23,6,16,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n///---\\\\\\\n:L<%^E!@\n1:Current.vdf\n9:Current\n15:0,0,0,0,0,0\n19:100,0\n27:0,\n34:0,\n4:Time\n5:Tickets pendientes\n35:Date\n36:YYYY-MM-DD\n37:2000\n38:1\n39:1\n40:0\n41:0\n42:1\n24:0\n25:24\n26:24\n',
    '03_Gusano_CodeRed_logistico': '{UTF-8}\nSusceptibles=\n\tINTEG (\n\t\t\t-Infecciones-Parcheo de susceptibles,\n\t\t\t\tHosts vulnerables-Infectados iniciales)\n\t~\thosts\n\t~\t\n\t|\n\nInfectados=\n\tINTEG (\n\t\t\tInfecciones-Parcheo de infectados,\n\t\t\t\tInfectados iniciales)\n\t~\thosts\n\t~\tNivel: servidores IIS comprometidos.\n\t|\n\nProtegidos=\n\tINTEG (\n\t\t\tParcheo de susceptibles+Parcheo de infectados,\n\t\t\t\t0)\n\t~\thosts\n\t~\t\n\t|\n\nInfecciones=\n\tTasa de contacto*Susceptibles*Infectados/Hosts vulnerables\n\t~\thosts/hora\n\t~\tCrecimiento logístico: R al inicio, B al agotarse los susceptibles.\n\t|\n\nParcheo de susceptibles=\n\tTasa de parcheo*Susceptibles*STEP(1, Inicio del parcheo)\n\t~\thosts/hora\n\t~\t\n\t|\n\nParcheo de infectados=\n\tTasa de parcheo*Infectados*STEP(1, Inicio del parcheo)\n\t~\thosts/hora\n\t~\t\n\t|\n\nTasa de contacto=\n\t1.33\n\t~\t1/hora\n\t~\tCalibrada: pico de unos 2 000 hosts por minuto (CAIDA, 2001).\n\t|\n\nHosts vulnerables=\n\t360000\n\t~\thosts\n\t~\tMás de 359 000 hosts infectados el 19 de julio de 2001 (CAIDA).\n\t|\n\nInfectados iniciales=\n\t30\n\t~\thosts\n\t~\t\n\t|\n\nTasa de parcheo=\n\t0\n\t~\t1/hora\n\t~\tEscenario: 0.5 desde la hora 4 o 6.\n\t|\n\nInicio del parcheo=\n\t4\n\t~\thora\n\t~\t\n\t|\n\n********************************************************\n\t.Control\n********************************************************~\n\t\tSimulation Control Parameters\n\t|\n\nFINAL TIME  = 16\n\t~\thora\n\t~\tTiempo final de la simulación.\n\t|\n\nINITIAL TIME  = 0\n\t~\thora\n\t~\tTiempo inicial de la simulación.\n\t|\n\nSAVEPER  = \n\tTIME STEP\n\t~\thora [0,?]\n\t~\tFrecuencia de guardado.\n\t|\n\nTIME STEP  = 0.0625\n\t~\thora [0,?]\n\t~\tPaso de integración (método de Euler).\n\t|\n\n\\\\\\---/// Sketch information - do not modify anything except names\nV300  Do not put anything below this section - it will be ignored\n*View 1\n$192-192-192,0,Times New Roman|12||0-0-0|0-0-0|0-0-255|-1--1--1|-1--1--1|96,96,100,0\n10,1,Susceptibles,160,220,46,26,3,3,0,0,0,0,0,0\n10,2,Infectados,420,220,46,26,3,3,0,0,0,0,0,0\n10,3,Protegidos,300,420,46,26,3,3,0,0,0,0,0,0\n1,4,6,2,4,0,0,22,0,0,0,-1--1--1,,1|(349,220)|\n1,5,6,1,100,0,0,22,0,0,0,-1--1--1,,1|(205,220)|\n11,6,48,290,220,6,8,34,3,0,0,1,0,0,0\n10,7,Infecciones,290,242,44,12,40,3,0,0,-1,0,0,0\n10,8,Tasa de contacto,290,110,40,12,8,3,0,0,0,0,0,0\n10,9,Hosts vulnerables,170,90,40,12,8,3,0,0,0,0,0,0\n10,10,Infectados iniciales,470,100,40,12,8,3,0,0,0,0,0,0\n10,11,Tasa de parcheo,560,330,40,12,8,3,0,0,0,0,0,0\n10,12,Inicio del parcheo,560,420,40,12,8,3,0,0,0,0,0,0\n10,13,Parcheo de susceptibles,150,330,40,12,8,3,0,0,0,0,0,0\n10,14,Parcheo de infectados,430,330,40,12,8,3,0,0,0,0,0,0\n1,15,8,6,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,16,9,6,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,17,1,6,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,18,2,6,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,19,11,14,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,20,11,13,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,21,12,14,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,22,1,13,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,23,2,14,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n///---\\\\\\\n:L<%^E!@\n1:Current.vdf\n9:Current\n15:0,0,0,0,0,0\n19:100,0\n27:0,\n34:0,\n4:Time\n5:Susceptibles\n35:Date\n36:YYYY-MM-DD\n37:2000\n38:1\n39:1\n40:0\n41:0\n42:1\n24:0\n25:16\n26:16\n',
    '04_Autoescalado_segundo_orden': '{UTF-8}\nInstancias en arranque=\n\tINTEG (\n\t\t\tSolicitud de instancias-Instancias listas,\n\t\t\t\t0)\n\t~\tinstancias\n\t~\tNivel 1: máquinas pedidas que aún no atienden tráfico.\n\t|\n\nInstancias activas=\n\tINTEG (\n\t\t\tInstancias listas-Retiro de instancias,\n\t\t\t\tDemanda base/(Capacidad por instancia*Utilizacion objetivo))\n\t~\tinstancias\n\t~\tNivel 2: máquinas que atienden tráfico.\n\t|\n\nUtilizacion medida=\n\tINTEG (\n\t\t\t(Utilizacion real-Utilizacion medida)/Tiempo de medicion,\n\t\t\t\tUtilizacion objetivo)\n\t~\tDmnl\n\t~\tMétrica suavizada (retardo de información, como la ventana del HPA).\n\t|\n\nInstancias listas=\n\tInstancias en arranque/Tiempo de arranque\n\t~\tinstancias/minuto\n\t~\t\n\t|\n\nBrecha de instancias=\n\tInstancias deseadas-Instancias activas-Considerar instancias en arranque*Instancias en arranque\n\t~\tinstancias\n\t~\t\n\t|\n\nSolicitud de instancias=\n\tMAX(Brecha de instancias/Tiempo de ajuste, 0)\n\t~\tinstancias/minuto\n\t~\t\n\t|\n\nRetiro de instancias=\n\tMAX(-Brecha de instancias/Tiempo de ajuste, 0)\n\t~\tinstancias/minuto\n\t~\t\n\t|\n\nInstancias deseadas=\n\tInstancias activas*Utilizacion medida/Utilizacion objetivo\n\t~\tinstancias\n\t~\tRegla del HPA de Kubernetes: réplicas · métrica actual / métrica deseada.\n\t|\n\nUtilizacion real=\n\tDemanda/(Capacidad por instancia*Instancias activas)\n\t~\tDmnl\n\t~\t\n\t|\n\nDemanda=\n\tDemanda base+(Demanda pico-Demanda base)*PULSE(Inicio del pico, Duracion del pico)\n\t~\tsolicitudes/segundo\n\t~\t\n\t|\n\nDemanda base=\n\t600\n\t~\tsolicitudes/segundo\n\t~\t\n\t|\n\nDemanda pico=\n\t1500\n\t~\tsolicitudes/segundo\n\t~\tCampaña o día sin IVA.\n\t|\n\nInicio del pico=\n\t10\n\t~\tminuto\n\t~\t\n\t|\n\nDuracion del pico=\n\t60\n\t~\tminuto\n\t~\t\n\t|\n\nCapacidad por instancia=\n\t100\n\t~\tsolicitudes/segundo/instancia\n\t~\t\n\t|\n\nUtilizacion objetivo=\n\t0.7\n\t~\tDmnl\n\t~\t\n\t|\n\nTiempo de ajuste=\n\t1\n\t~\tminuto\n\t~\t\n\t|\n\nTiempo de arranque=\n\t4\n\t~\tminuto\n\t~\t\n\t|\n\nTiempo de medicion=\n\t2\n\t~\tminuto\n\t~\t\n\t|\n\nConsiderar instancias en arranque=\n\t0\n\t~\tDmnl\n\t~\t0 = regla ingenua (oscila); 1 = cuenta las instancias en camino.\n\t|\n\n********************************************************\n\t.Control\n********************************************************~\n\t\tSimulation Control Parameters\n\t|\n\nFINAL TIME  = 120\n\t~\tminuto\n\t~\tTiempo final de la simulación.\n\t|\n\nINITIAL TIME  = 0\n\t~\tminuto\n\t~\tTiempo inicial de la simulación.\n\t|\n\nSAVEPER  = \n\tTIME STEP\n\t~\tminuto [0,?]\n\t~\tFrecuencia de guardado.\n\t|\n\nTIME STEP  = 0.0625\n\t~\tminuto [0,?]\n\t~\tPaso de integración (método de Euler).\n\t|\n\n\\\\\\---/// Sketch information - do not modify anything except names\nV300  Do not put anything below this section - it will be ignored\n*View 1\n$192-192-192,0,Times New Roman|12||0-0-0|0-0-0|0-0-255|-1--1--1|-1--1--1|96,96,100,0\n12,1,48,90,250,10,8,0,3,0,0,-1,0,0,0\n10,2,Instancias en arranque,300,250,46,26,3,3,0,0,0,0,0,0\n10,3,Instancias activas,540,250,46,26,3,3,0,0,0,0,0,0\n12,4,48,740,250,10,8,0,3,0,0,-1,0,0,0\n1,5,7,2,4,0,0,22,0,0,0,-1--1--1,,1|(206,250)|\n1,6,7,1,100,0,0,22,0,0,0,-1--1--1,,1|(122,250)|\n11,7,48,177,250,6,8,34,3,0,0,1,0,0,0\n10,8,Solicitud de instancias,177,272,44,12,40,3,0,0,-1,0,0,0\n1,9,11,3,4,0,0,22,0,0,0,-1--1--1,,1|(446,250)|\n1,10,11,2,100,0,0,22,0,0,0,-1--1--1,,1|(368,250)|\n11,11,48,420,250,6,8,34,3,0,0,1,0,0,0\n10,12,Instancias listas,420,272,44,12,40,3,0,0,-1,0,0,0\n1,13,15,4,4,0,0,22,0,0,0,-1--1--1,,1|(682,250)|\n1,14,15,3,100,0,0,22,0,0,0,-1--1--1,,1|(608,250)|\n11,15,48,658,250,6,8,34,3,0,0,1,0,0,0\n10,16,Retiro de instancias,658,272,44,12,40,3,0,0,-1,0,0,0\n10,17,Brecha de instancias,300,400,40,12,8,3,0,0,0,0,0,0\n10,18,Instancias deseadas,470,420,40,12,8,3,0,0,0,0,0,0\n10,19,Utilizacion real,640,420,40,12,8,3,0,0,0,0,0,0\n10,20,Utilizacion medida,560,500,40,12,8,3,0,0,0,0,0,0\n10,21,Demanda,760,380,40,12,8,3,0,0,0,0,0,0\n10,22,Demanda base,820,470,40,12,8,3,0,0,0,0,0,0\n10,23,Demanda pico,840,320,40,12,8,3,0,0,0,0,0,0\n10,24,Inicio del pico,860,420,40,12,8,3,0,0,0,0,0,0\n10,25,Duracion del pico,820,520,40,12,8,3,0,0,0,0,0,0\n10,26,Capacidad por instancia,700,520,40,12,8,3,0,0,0,0,0,0\n10,27,Utilizacion objetivo,380,520,40,12,8,3,0,0,0,0,0,0\n10,28,Tiempo de ajuste,160,380,40,12,8,3,0,0,0,0,0,0\n10,29,Tiempo de arranque,420,130,40,12,8,3,0,0,0,0,0,0\n10,30,Tiempo de medicion,700,590,40,12,8,3,0,0,0,0,0,0\n10,31,Considerar instancias en arranque,160,470,40,12,8,3,0,0,0,0,0,0\n1,32,17,7,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,33,17,15,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,34,28,7,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,35,28,15,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,36,2,11,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,37,29,11,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,38,18,17,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,39,3,17,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,40,2,17,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,41,31,17,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,42,3,18,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,43,20,18,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,44,27,18,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,45,21,19,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,46,26,19,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,47,3,19,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,48,19,20,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,49,30,20,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,50,22,21,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,51,23,21,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,52,24,21,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,53,25,21,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n///---\\\\\\\n:L<%^E!@\n1:Current.vdf\n9:Current\n15:0,0,0,0,0,0\n19:100,0\n27:0,\n34:0,\n4:Time\n5:Instancias en arranque\n35:Date\n36:YYYY-MM-DD\n37:2000\n38:1\n39:1\n40:0\n41:0\n42:1\n24:0\n25:120\n26:120\n',
    '05_Proyecto_Ley_de_Brooks': '{UTF-8}\nTareas pendientes=\n\tINTEG (\n\t\t\t-Avance,\n\t\t\t\tTamano del proyecto)\n\t~\ttareas\n\t~\t\n\t|\n\nNovatos=\n\tINTEG (\n\t\t\tContratacion-Asimilacion,\n\t\t\t\t0)\n\t~\tpersonas\n\t~\t\n\t|\n\nExpertos=\n\tINTEG (\n\t\t\tAsimilacion,\n\t\t\t\tExpertos iniciales)\n\t~\tpersonas\n\t~\t\n\t|\n\nContratacion=\n\tPersonas a contratar*PULSE(Dia de contratacion, TIME STEP)/TIME STEP\n\t~\tpersonas/dia\n\t~\tIngreso de nuevos desarrolladores en un solo día.\n\t|\n\nAsimilacion=\n\tNovatos/Tiempo de asimilacion\n\t~\tpersonas/dia\n\t~\t\n\t|\n\nTamano del equipo=\n\tNovatos+Expertos\n\t~\tpersonas\n\t~\t\n\t|\n\nSobrecarga de comunicacion=\n\tMIN(0.9, Coeficiente de comunicacion*(Tamano del equipo/10)^2)\n\t~\tDmnl\n\t~\tBrooks: los canales de comunicación crecen como n(n-1)/2.\n\t|\n\nProductividad bruta=\n\tMAX(0, (Expertos-Tiempo de mentoria*Novatos)*Productividad experto+Novatos*Productividad novato)\n\t~\ttareas/dia\n\t~\tLos novatos consumen tiempo de los expertos.\n\t|\n\nAvance=\n\tIF THEN ELSE(Tareas pendientes>0, Productividad bruta*(1-Sobrecarga de comunicacion), 0)\n\t~\ttareas/dia\n\t~\t\n\t|\n\nTamano del proyecto=\n\t2000\n\t~\ttareas\n\t~\t\n\t|\n\nExpertos iniciales=\n\t10\n\t~\tpersonas\n\t~\t\n\t|\n\nPersonas a contratar=\n\t0\n\t~\tpersonas\n\t~\tEscenarios: 10 personas el día 0, 100 o 150.\n\t|\n\nDia de contratacion=\n\t150\n\t~\tdia\n\t~\t\n\t|\n\nTiempo de asimilacion=\n\t60\n\t~\tdia\n\t~\t\n\t|\n\nCoeficiente de comunicacion=\n\t0.08\n\t~\tDmnl\n\t~\t\n\t|\n\nTiempo de mentoria=\n\t0.5\n\t~\tDmnl\n\t~\tFracción de un experto que consume cada novato.\n\t|\n\nProductividad experto=\n\t1\n\t~\ttareas/persona/dia\n\t~\t\n\t|\n\nProductividad novato=\n\t0.2\n\t~\ttareas/persona/dia\n\t~\t\n\t|\n\n********************************************************\n\t.Control\n********************************************************~\n\t\tSimulation Control Parameters\n\t|\n\nFINAL TIME  = 400\n\t~\tdia\n\t~\tTiempo final de la simulación.\n\t|\n\nINITIAL TIME  = 0\n\t~\tdia\n\t~\tTiempo inicial de la simulación.\n\t|\n\nSAVEPER  = \n\tTIME STEP\n\t~\tdia [0,?]\n\t~\tFrecuencia de guardado.\n\t|\n\nTIME STEP  = 0.25\n\t~\tdia [0,?]\n\t~\tPaso de integración (método de Euler).\n\t|\n\n\\\\\\---/// Sketch information - do not modify anything except names\nV300  Do not put anything below this section - it will be ignored\n*View 1\n$192-192-192,0,Times New Roman|12||0-0-0|0-0-0|0-0-255|-1--1--1|-1--1--1|96,96,100,0\n12,1,48,130,230,10,8,0,3,0,0,-1,0,0,0\n10,2,Novatos,330,230,46,26,3,3,0,0,0,0,0,0\n10,3,Expertos,560,230,46,26,3,3,0,0,0,0,0,0\n10,4,Tareas pendientes,450,480,46,26,3,3,0,0,0,0,0,0\n12,5,48,700,480,10,8,0,3,0,0,-1,0,0,0\n1,6,8,2,4,0,0,22,0,0,0,-1--1--1,,1|(236,230)|\n1,7,8,1,100,0,0,22,0,0,0,-1--1--1,,1|(162,230)|\n11,8,48,212,230,6,8,34,3,0,0,1,0,0,0\n10,9,Contratacion,212,252,44,12,40,3,0,0,-1,0,0,0\n1,10,12,3,4,0,0,22,0,0,0,-1--1--1,,1|(466,230)|\n1,11,12,2,100,0,0,22,0,0,0,-1--1--1,,1|(398,230)|\n11,12,48,445,230,6,8,34,3,0,0,1,0,0,0\n10,13,Asimilacion,445,252,44,12,40,3,0,0,-1,0,0,0\n1,14,16,5,4,0,0,22,0,0,0,-1--1--1,,1|(642,480)|\n1,15,16,4,100,0,0,22,0,0,0,-1--1--1,,1|(518,480)|\n11,16,48,593,480,6,8,34,3,0,0,1,0,0,0\n10,17,Avance,593,502,44,12,40,3,0,0,-1,0,0,0\n10,18,Tamano del equipo,450,340,40,12,8,3,0,0,0,0,0,0\n10,19,Sobrecarga de comunicacion,620,380,40,12,8,3,0,0,0,0,0,0\n10,20,Productividad bruta,330,400,40,12,8,3,0,0,0,0,0,0\n10,21,Personas a contratar,120,120,40,12,8,3,0,0,0,0,0,0\n10,22,Dia de contratacion,240,110,40,12,8,3,0,0,0,0,0,0\n10,23,Tiempo de asimilacion,450,120,40,12,8,3,0,0,0,0,0,0\n10,24,Coeficiente de comunicacion,760,330,40,12,8,3,0,0,0,0,0,0\n10,25,Tiempo de mentoria,200,360,40,12,8,3,0,0,0,0,0,0\n10,26,Productividad experto,200,450,40,12,8,3,0,0,0,0,0,0\n10,27,Productividad novato,260,520,40,12,8,3,0,0,0,0,0,0\n10,28,Expertos iniciales,640,140,40,12,8,3,0,0,0,0,0,0\n10,29,Tamano del proyecto,330,580,40,12,8,3,0,0,0,0,0,0\n1,30,21,8,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,31,22,8,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,32,2,12,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,33,23,12,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,34,2,18,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,35,3,18,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,36,18,19,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,37,24,19,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,38,3,20,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,39,2,20,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,40,25,20,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,41,26,20,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,42,27,20,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,43,20,16,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,44,19,16,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,45,4,16,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n///---\\\\\\\n:L<%^E!@\n1:Current.vdf\n9:Current\n15:0,0,0,0,0,0\n19:100,0\n27:0,\n34:0,\n4:Time\n5:Tareas pendientes\n35:Date\n36:YYYY-MM-DD\n37:2000\n38:1\n39:1\n40:0\n41:0\n42:1\n24:0\n25:400\n26:400\n',
    '06_Ventilador_PI_segundo_orden': '{UTF-8}\nTemperatura servidor=\n\tINTEG (\n\t\t\tCambio de temperatura,\n\t\t\t\tTemperatura objetivo inicial)\n\t~\tC\n\t~\tNivel 1 del lazo cerrado.\n\t|\n\nError acumulado=\n\tINTEG (\n\t\t\tAcumulacion del error,\n\t\t\t\tVentilador inicial/Ki)\n\t~\tC*s\n\t~\tNivel 2: la acción integral del controlador PI.\n\t|\n\nCambio de temperatura=\n\t(Temperatura ambiente+Potencia*Resistencia termica-Efecto del ventilador*Velocidad del ventilador-Temperatura servidor)/Constante de tiempo\n\t~\tC/s\n\t~\t\n\t|\n\nError=\n\tTemperatura servidor-Temperatura objetivo\n\t~\tC\n\t~\t\n\t|\n\nTemperatura objetivo=\n\tTemperatura objetivo inicial+STEP(Nuevo objetivo-Temperatura objetivo inicial, Momento del cambio)\n\t~\tC\n\t~\t\n\t|\n\nSenal del controlador=\n\tKp*Error+Ki*Error acumulado\n\t~\t%\n\t~\t\n\t|\n\nVelocidad del ventilador=\n\tMAX(0, MIN(100, Senal del controlador))\n\t~\t%\n\t~\t\n\t|\n\nAcumulacion del error=\n\tIF THEN ELSE(Senal del controlador>=100 :AND: Error>0, 0, IF THEN ELSE(Senal del controlador<=0 :AND: Error<0, 0, Error))\n\t~\tC\n\t~\tAnti-windup: deja de integrar si el ventilador está saturado.\n\t|\n\nVentilador inicial=\n\t(Temperatura ambiente+Potencia*Resistencia termica-Temperatura objetivo inicial)/Efecto del ventilador\n\t~\t%\n\t~\t\n\t|\n\nPotencia=\n\tPotencia base+STEP(Potencia extra, Momento de la carga)\n\t~\tW\n\t~\t\n\t|\n\nKp=\n\t1.5\n\t~\t%/C\n\t~\tAgresivo: 0.3 · Bien ajustado: 1.5 · Lento: 1\n\t|\n\nKi=\n\t0.1\n\t~\t%/(C*s)\n\t~\tAgresivo: 0.15 · Bien ajustado: 0.1 · Lento: 0.01\n\t|\n\nTemperatura objetivo inicial=\n\t60\n\t~\tC\n\t~\t\n\t|\n\nNuevo objetivo=\n\t50\n\t~\tC\n\t~\t\n\t|\n\nMomento del cambio=\n\t50\n\t~\ts\n\t~\t\n\t|\n\nTemperatura ambiente=\n\t25\n\t~\tC\n\t~\t\n\t|\n\nResistencia termica=\n\t0.4\n\t~\tC/W\n\t~\t\n\t|\n\nEfecto del ventilador=\n\t0.3\n\t~\tC/%\n\t~\t\n\t|\n\nConstante de tiempo=\n\t40\n\t~\ts\n\t~\t\n\t|\n\nPotencia base=\n\t120\n\t~\tW\n\t~\t\n\t|\n\nPotencia extra=\n\t0\n\t~\tW\n\t~\tPerturbación opcional.\n\t|\n\nMomento de la carga=\n\t300\n\t~\ts\n\t~\t\n\t|\n\n********************************************************\n\t.Control\n********************************************************~\n\t\tSimulation Control Parameters\n\t|\n\nFINAL TIME  = 600\n\t~\ts\n\t~\tTiempo final de la simulación.\n\t|\n\nINITIAL TIME  = 0\n\t~\ts\n\t~\tTiempo inicial de la simulación.\n\t|\n\nSAVEPER  = \n\tTIME STEP\n\t~\ts [0,?]\n\t~\tFrecuencia de guardado.\n\t|\n\nTIME STEP  = 0.125\n\t~\ts [0,?]\n\t~\tPaso de integración (método de Euler).\n\t|\n\n\\\\\\---/// Sketch information - do not modify anything except names\nV300  Do not put anything below this section - it will be ignored\n*View 1\n$192-192-192,0,Times New Roman|12||0-0-0|0-0-0|0-0-255|-1--1--1|-1--1--1|96,96,100,0\n12,1,48,110,230,10,8,0,3,0,0,-1,0,0,0\n10,2,Temperatura servidor,330,230,46,26,3,3,0,0,0,0,0,0\n12,3,48,330,520,10,8,0,3,0,0,-1,0,0,0\n10,4,Error acumulado,580,470,46,26,3,3,0,0,0,0,0,0\n1,5,7,2,4,0,0,22,0,0,0,-1--1--1,,1|(236,230)|\n1,6,7,1,100,0,0,22,0,0,0,-1--1--1,,1|(142,230)|\n11,7,48,202,230,6,8,34,3,0,0,1,0,0,0\n10,8,Cambio de temperatura,202,252,44,12,40,3,0,0,-1,0,0,0\n1,9,11,4,4,0,0,22,0,0,0,-1--1--1,,1|(486,470)|\n1,10,11,3,100,0,0,22,0,0,0,-1--1--1,,1|(374,470)|\n11,11,48,443,470,6,8,34,3,0,0,1,0,0,0\n10,12,Acumulacion del error,443,492,44,12,40,3,0,0,-1,0,0,0\n10,13,Error,530,320,40,12,8,3,0,0,0,0,0,0\n10,14,Temperatura objetivo,600,210,40,12,8,3,0,0,0,0,0,0\n10,15,Senal del controlador,760,380,40,12,8,3,0,0,0,0,0,0\n10,16,Velocidad del ventilador,520,110,40,12,8,3,0,0,0,0,0,0\n10,17,Ventilador inicial,760,530,40,12,8,3,0,0,0,0,0,0\n10,18,Potencia,140,110,40,12,8,3,0,0,0,0,0,0\n10,19,Kp,860,300,40,12,8,3,0,0,0,0,0,0\n10,20,Ki,860,460,40,12,8,3,0,0,0,0,0,0\n10,21,Temperatura objetivo inicial,760,130,40,12,8,3,0,0,0,0,0,0\n10,22,Nuevo objetivo,760,70,40,12,8,3,0,0,0,0,0,0\n10,23,Momento del cambio,640,60,40,12,8,3,0,0,0,0,0,0\n10,24,Temperatura ambiente,90,340,40,12,8,3,0,0,0,0,0,0\n10,25,Resistencia termica,190,380,40,12,8,3,0,0,0,0,0,0\n10,26,Efecto del ventilador,330,60,40,12,8,3,0,0,0,0,0,0\n10,27,Constante de tiempo,230,330,40,12,8,3,0,0,0,0,0,0\n10,28,Potencia base,60,60,40,12,8,3,0,0,0,0,0,0\n10,29,Potencia extra,160,30,40,12,8,3,0,0,0,0,0,0\n10,30,Momento de la carga,260,50,40,12,8,3,0,0,0,0,0,0\n1,31,2,13,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,32,14,13,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,33,13,15,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,34,4,15,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,35,19,15,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,36,20,15,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,37,15,16,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,38,16,7,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,39,2,7,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,40,18,7,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,41,24,7,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,42,25,7,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,43,26,7,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,44,27,7,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,45,13,11,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,46,15,11,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,47,21,14,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,48,22,14,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,49,23,14,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,50,28,18,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,51,29,18,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n1,52,30,18,0,0,0,0,0,128,0,-1--1--1,,1|(0,0)|\n///---\\\\\\\n:L<%^E!@\n1:Current.vdf\n9:Current\n15:0,0,0,0,0,0\n19:100,0\n27:0,\n34:0,\n4:Time\n5:Temperatura servidor\n35:Date\n36:YYYY-MM-DD\n37:2000\n38:1\n39:1\n40:0\n41:0\n42:1\n24:0\n25:600\n26:600\n',
}
os.makedirs("vensim", exist_ok=True)
for nombre, texto in MDL.items():
    with open(f"vensim/{nombre}.mdl", "w", encoding="utf-8") as fh: fh.write(texto)
print("Modelos escritos:", sorted(os.listdir("vensim")))

### Intérprete mínimo de Vensim (respaldo)
Si PySD no está instalado, esta función lee las ecuaciones del `.mdl` y las simula con Euler, igual que Vensim. Admite `INTEG`, `STEP`, `PULSE`, `IF THEN ELSE`, `MIN`, `MAX`, `EXP`, `SQRT`, `:AND:` y `:OR:`.

In [ ]:
import re, math
import numpy as np
import pandas as pd

def leer_mdl(texto):
    """Lee las ecuaciones de un archivo .mdl de Vensim (ignora el diagrama). Devuelve (ecuaciones, control)."""
    cuerpo = texto.split("\\\\\\---///")[0].replace("{UTF-8}", "")
    eqs, control = {}, {}
    for bloque in cuerpo.split("|"):
        partes = bloque.split("~")
        if len(partes) < 2 or "=" not in partes[0]: continue
        izq, der = partes[0].split("=", 1)
        nombre = " ".join(izq.split()); expr = " ".join(der.split())
        if not nombre or nombre.startswith("*"): continue
        if nombre in ("FINAL TIME", "INITIAL TIME", "TIME STEP", "SAVEPER"): control[nombre] = expr
        else: eqs[nombre] = expr
    return eqs, control

def simular_mdl(texto, params=None, final_time=None, time_step=None):
    """Intérprete mínimo de Vensim (método de Euler). Admite INTEG, STEP, PULSE, IF THEN ELSE, MIN, MAX, EXP, SQRT, ABS, LN, :AND:, :OR:."""
    eqs, ctl = leer_mdl(texto)
    for k, v in (params or {}).items(): eqs[k] = str(v)
    dt = float(time_step or eval(ctl.get("TIME STEP", "1"))); T = float(final_time or eval(ctl.get("FINAL TIME", "10")))
    ident = {n: "v_" + re.sub(r"\W", "_", n) for n in eqs}; ident["TIME STEP"] = "DT"; ident["Time"] = "TIME"
    def tr(e):
        e = e.replace("IF THEN ELSE", "IFTE").replace(":AND:", " and ").replace(":OR:", " or ").replace("^", "**")
        for n in sorted(ident, key=len, reverse=True): e = re.sub(r"(?<![\w])" + re.escape(n) + r"(?![\w])", ident[n], e)
        return re.sub(r"(?<![<>=!])=(?!=)", "==", e)
    def split2(s):
        d = 0
        for i, ch in enumerate(s):
            d += ch == "("; d -= ch == ")"
            if ch == "," and d == 0: return s[:i], s[i + 1:]
    niveles = {}
    for n, e in eqs.items():
        m = re.match(r"INTEG\s*\((.*)\)\s*$", e)
        if m: niveles[n] = tuple(tr(p) for p in split2(m.group(1)))
    aux = {n: tr(e) for n, e in eqs.items() if n not in niveles}
    base = dict(IFTE=lambda c, a, b: a if c else b, MIN=min, MAX=max, EXP=math.exp, SQRT=math.sqrt, ABS=abs, LN=math.log, DT=dt)
    def evaluar(t, st, inicial=False):
        e = dict(base); e["TIME"] = t
        e["STEP"] = lambda h, t0: h if t >= t0 - 1e-9 else 0
        e["PULSE"] = lambda s0, w: 1 if s0 - 1e-9 <= t < s0 + w - 1e-9 else 0
        for n, v in st.items(): e[ident[n]] = v
        pend = dict(aux); pend_n = dict(niveles) if inicial else {}
        for _ in range(100):
            for n in list(pend):
                try: e[ident[n]] = eval(pend[n], {}, e); del pend[n]
                except NameError: pass
            for n in list(pend_n):
                try: e[ident[n]] = eval(pend_n[n][1], {}, e); st[n] = e[ident[n]]; del pend_n[n]
                except NameError: pass
            if not pend and not pend_n: break
        if pend or pend_n: raise RuntimeError("Variables sin resolver: " + ", ".join(list(pend) + list(pend_n)))
        return e
    st = {}; evaluar(0.0, st, inicial=True)
    filas = []; n_pasos = int(round(T / dt))
    for k in range(n_pasos + 1):
        t = k * dt; e = evaluar(t, st)
        filas.append({"Time": t, **{n: st[n] for n in niveles}, **{n: e[ident[n]] for n in aux}})
        if k < n_pasos: st = {n: st[n] + dt * eval(niveles[n][0], {}, e) for n in st}
    return pd.DataFrame(filas).set_index("Time")


def correr(nombre, params=None, **kw):
    """Ejecuta un .mdl con PySD si está disponible; si no, con el intérprete mínimo. Devuelve un DataFrame."""
    ruta = f"vensim/{nombre}.mdl"
    if pysd is not None:
        try:
            modelo = pysd.read_vensim(ruta)
            return modelo.run(params=params or {}, **({"final_time": kw["final_time"]} if "final_time" in kw else {}))
        except Exception as e:
            print("PySD no pudo con", nombre, "→ se usa el intérprete mínimo:", type(e).__name__)
    return simular_mdl(open(ruta, encoding="utf-8").read(), params, **kw)

base_cpu = correr("01_CPU_primer_orden")
print("Temperatura final de la CPU según el .mdl:", round(float(base_cpu["Temperatura CPU"].iloc[-1]), 3), "°C")

---
## 4. Caso 1 · Temperatura de la CPU (primer orden lineal)
$\dfrac{dT}{dt} = \dfrac{P}{C_{th}} - \dfrac{T - T_a}{R_{th}\,C_{th}}$, con $\tau = R_{th} C_{th} = 20$ s. El mismo modelo en Python a mano y con el `.mdl`, más tres escenarios de refrigeración.

In [ ]:
def cpu(Rth=0.4, Cth=50.0, Ta=25.0, P0=15.0, P1=95.0, t_step=30.0, T=200.0, dt=0.5):
    P = lambda t: P1 if t >= t_step else P0
    return euler(lambda t, x: [P(t) / Cth - (x[0] - Ta) / (Rth * Cth)], [Ta + P0 * Rth], T, dt, ["T"])
esc = {"Disipador limpio (0,4 °C/W)": dict(Rth=0.4), "Disipador sucio (0,75 °C/W)": dict(Rth=0.75), "Refrigeración líquida (0,25 °C/W)": dict(Rth=0.25, Cth=80)}
for (nom, p), c in zip(esc.items(), [COL["cyan"], COL["rose"], COL["indigo"]]):
    d = cpu(**p); plt.plot(d.index, d["T"], color=c, lw=2.3, label=f"{nom}: {d['T'].iloc[-1]:.1f} °C")
vm = correr("01_CPU_primer_orden"); plt.plot(vm.index, vm["Temperatura CPU"], "k:", lw=2, label="mismo caso desde el .mdl")
plt.axhline(90, ls="--", color="gray", label="umbral de throttling (90 °C)"); plt.title("Temperatura de la CPU"); plt.xlabel("segundos"); plt.ylabel("°C"); plt.legend(); plt.show()
print("Diferencia máxima Python vs .mdl:", float(np.abs(cpu()["T"].values - vm["Temperatura CPU"].values).max()))

## 5. Caso 2 · Cola de tickets (primer orden lineal y ley de Little)
$\dfrac{dB}{dt} = \lambda(t) - \dfrac{B}{\tau}$. En equilibrio $B^* = \lambda\tau$ y la espera es $W = B/\lambda = \tau$ (Little, 1961). Escenarios del incidente (55, 70 o 100 tickets/h) y una política (más agentes: τ = 1,4 h).

In [ ]:
escen = {"Optimista (55/h)": {"Llegadas en incidente": 55}, "Base (70/h)": {}, "Pesimista (100/h)": {"Llegadas en incidente": 100}, "Base + más agentes": {"Tiempo de resolucion": 1.4}}
res = {}
for (nom, p), c in zip(escen.items(), [COL["green"], COL["cyan"], COL["rose"], COL["indigo"]]):
    d = correr("02_Cola_tickets_primer_orden", p); res[nom] = d; plt.plot(d.index, d["Tickets pendientes"], color=c, lw=2.3, label=nom)
plt.title("Tickets pendientes por escenario"); plt.xlabel("hora"); plt.ylabel("tickets"); plt.legend(); plt.show()
pd.DataFrame({k: {"Tickets al final": v["Tickets pendientes"].iloc[-1], "Espera estimada (h)": v["Tiempo de espera estimado"].iloc[-1]} for k, v in res.items()}).T.round(2)

## 6. Caso 3 · Gusano Code-Red (primer orden no lineal)
Code-Red v2 infectó más de 359 000 hosts el 19 de julio de 2001 en menos de 14 horas, con picos de más de 2 000 hosts por minuto (CAIDA). Modelo logístico con β = 1,33/h y escenarios de parcheo.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
for nom, p, c in [("Sin parches", {}, COL["rose"]), ("Parches desde la hora 6", {"Tasa de parcheo": 0.5, "Inicio del parcheo": 6}, COL["amber"]), ("Parches desde la hora 4", {"Tasa de parcheo": 0.5, "Inicio del parcheo": 4}, COL["cyan"])]:
    d = correr("03_Gusano_CodeRed_logistico", p); ax[0].plot(d.index, d["Infectados"] / 1000, color=c, lw=2.3, label=f"{nom}: máx {d['Infectados'].max():,.0f}")
    if nom == "Sin parches": ax[1].plot(d.index, d["Infecciones"] / 60, color=c, lw=2.3)
ax[0].set(title="Hosts infectados (miles)", xlabel="horas desde las 10:00 UTC"); ax[0].legend()
ax[1].axhline(2000, ls="--", color="gray", label="dato CAIDA: > 2 000 por minuto"); ax[1].set(title="Nuevas infecciones por minuto (sin parches)", xlabel="horas"); ax[1].legend(); plt.tight_layout(); plt.show()

## 7. Caso 4 · Autoescalado en la nube (segundo orden y superior)
Regla del Horizontal Pod Autoscaler: réplicas deseadas = réplicas actuales × (métrica actual / métrica deseada). Las instancias tardan 4 min en arrancar y la métrica se suaviza en 2 min. Comparamos la regla ingenua con la que **cuenta las instancias en camino**.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4)); kpi = {}
for nom, c, col in [("Regla ingenua", 0, COL["rose"]), ("Cuenta las instancias en camino", 1, COL["cyan"])]:
    d = correr("04_Autoescalado_segundo_orden", {"Considerar instancias en arranque": c}); dt = d.index[1] - d.index[0]
    ax[0].plot(d.index, d["Instancias activas"], color=col, lw=2.3, label=nom); ax[1].plot(d.index, d["Utilizacion real"], color=col, lw=2)
    kpi[nom] = {"Máximo de instancias": d["Instancias activas"].max(), "Minutos con U > 90 %": (d["Utilizacion real"] > 0.9).sum() * dt, "Costo (instancia-min)": d["Instancias activas"].sum() * dt}
ax[0].plot(d.index, d["Demanda"] / 70, "--", color="gray", label="instancias necesarias"); ax[0].set(title="Instancias activas", xlabel="minutos"); ax[0].legend()
ax[1].axhline(0.9, ls="--", color="gray"); ax[1].set(title="Utilización real", xlabel="minutos"); plt.tight_layout(); plt.show()
pd.DataFrame(kpi).T.round(1)

## 8. Caso 5 · Proyecto de software y ley de Brooks (orden 3)
Novatos que tardan 60 días en volverse expertos, consumen mentoría y aumentan la comunicación ($0{,}08\,(n/10)^2$). ¿Qué pasa si se contratan 10 personas en distintos momentos?

In [ ]:
def fin(d): return (d["Tareas pendientes"] > 0).sum() * (d.index[1] - d.index[0])
casos5 = {"Sin contratar": {}, "10 personas el día 0": {"Personas a contratar": 10, "Dia de contratacion": 0}, "10 personas el día 100": {"Personas a contratar": 10, "Dia de contratacion": 100}, "10 personas el día 150": {"Personas a contratar": 10, "Dia de contratacion": 150}}
for (nom, p), c in zip(casos5.items(), [COL["navy"], COL["green"], COL["amber"], COL["rose"]]):
    d = correr("05_Proyecto_Ley_de_Brooks", p); plt.plot(d.index, d["Tareas pendientes"].clip(lower=0), color=c, lw=2.3, label=f"{nom}: termina el día {fin(d):.0f}")
plt.title("Tareas pendientes"); plt.xlabel("días"); plt.legend(); plt.show()

## 9. Caso 6 · Control PI del ventilador (segundo orden en lazo cerrado)
Con planta $\tau = 40$ s y ganancia $K = 0{,}3$ °C por punto de ventilador, el lazo cerrado cumple $\tau s^2 + (1 + K K_p)s + K K_i = 0$, de donde $\omega_n = \sqrt{K K_i/\tau}$ y $\zeta = (1 + K K_p)/(2\sqrt{\tau K K_i})$. La teoría predice el sobrepaso antes de simular.

In [ ]:
filas = []
for nom, kp, ki, c in [("Agresivo", 0.3, 0.15, COL["rose"]), ("Bien ajustado", 1.5, 0.1, COL["cyan"]), ("Lento", 1.0, 0.01, COL["slate"])]:
    d = correr("06_Ventilador_PI_segundo_orden", {"Kp": kp, "Ki": ki}); K, tau = 0.3, 40
    wn = math.sqrt(K * ki / tau); z = (1 + K * kp) / (2 * math.sqrt(tau * K * ki)); m = metricas(z, wn)
    plt.plot(d.index, d["Temperatura servidor"], color=c, lw=2.3, label=f"{nom}: ζ = {z:.2f}")
    filas.append([nom, kp, ki, round(z, 2), round(wn, 4), round(100 * m["Mp"], 1), round(100 * (50 - d["Temperatura servidor"].min()) / 10, 1)])
plt.axhline(50, ls="--", color="gray"); plt.title("Temperatura del servidor tras cambiar el objetivo de 60 a 50 °C"); plt.xlabel("segundos"); plt.ylabel("°C"); plt.legend(); plt.show()
pd.DataFrame(filas, columns=["Ajuste", "Kp", "Ki", "ζ", "ωn", "Mp teórico (%)", "Mp simulado (%)"])

**Ejercicio guiado:** elija $K_p$ y $K_i$ para $\zeta = 0{,}7$ y $\omega_n = 0{,}03$ rad/s despejando las fórmulas, y verifíquelo con `correr(...)`.

In [ ]:
zeta_obj, wn_obj, K, tau = 0.7, 0.03, 0.3, 40
Ki = wn_obj ** 2 * tau / K; Kp = (2 * zeta_obj * wn_obj * tau - 1) / K
d = correr("06_Ventilador_PI_segundo_orden", {"Kp": round(Kp, 3), "Ki": round(Ki, 4)})
print(f"Kp = {Kp:.3f}, Ki = {Ki:.4f} → sobrepaso simulado {100 * (50 - d['Temperatura servidor'].min()) / 10:.1f} % (teórico {100 * metricas(zeta_obj, wn_obj)['Mp']:.1f} %)")
print("La diferencia con la fórmula se debe al cero que agrega la acción proporcional del PI: la fórmula es exacta solo para el segundo orden puro.")

---
## 10. Análisis de escenarios
### 10.1 Barrido de una política: ¿cuándo deja de convenir contratar?

In [ ]:
dias = list(range(0, 201, 10)); fines = [fin(correr("05_Proyecto_Ley_de_Brooks", {"Personas a contratar": 10, "Dia de contratacion": dd})) for dd in dias]
sin = fin(correr("05_Proyecto_Ley_de_Brooks"))
plt.plot(dias, fines, "o-", color=COL["cyan"], lw=2.3, label="contratando 10 personas"); plt.axhline(sin, ls="--", color=COL["rose"], label=f"sin contratar ({sin:.0f} días)")
plt.title("Día de terminación según el día de contratación"); plt.xlabel("día de contratación"); plt.ylabel("día de terminación"); plt.legend(); plt.show()
corte = next(dd for dd, ff in zip(dias, fines) if ff > sin); print("Contratar a partir del día", corte, "retrasa el proyecto (ley de Brooks).")

### 10.2 Sensibilidad local (tornado) del sobrepaso en el autoescalado

In [ ]:
def sobrepaso(p):
    d = correr("04_Autoescalado_segundo_orden", p); obj = 1500 / (p.get("Capacidad por instancia", 100) * p.get("Utilizacion objetivo", 0.7)); return 100 * (d["Instancias activas"].max() - obj) / obj
base = sobrepaso({}); torn = []
for nom, v in [("Tiempo de ajuste", 1.0), ("Tiempo de arranque", 4.0), ("Tiempo de medicion", 2.0), ("Utilizacion objetivo", 0.7), ("Capacidad por instancia", 100.0)]:
    torn.append([nom, sobrepaso({nom: v * 0.8}) - base, sobrepaso({nom: v * 1.2}) - base])
torn = pd.DataFrame(torn, columns=["parámetro", "−20 %", "+20 %"]).set_index("parámetro"); torn = torn.reindex(torn.abs().max(axis=1).sort_values().index)
y = np.arange(len(torn)); plt.barh(y, torn["−20 %"], color=COL["cyan"], label="−20 %"); plt.barh(y, torn["+20 %"], color=COL["rose"], label="+20 %"); plt.yticks(y, torn.index); plt.axvline(0, color="k")
plt.title(f"Cambio del sobrepaso (base {base:.0f} %)"); plt.xlabel("puntos porcentuales"); plt.legend(); plt.show()

### 10.3 Monte Carlo: horas hasta infectar el 90 % (gusano)
Vensim PLE no trae Monte Carlo (requiere PLE Plus o superior); en Python es un bucle. Usamos la solución analítica de la logística: $t_{90} = \ln\!\big(9 (N - I_0)/I_0\big)/\beta$, y además 50 corridas del `.mdl` para comprobar.

In [ ]:
rng = np.random.default_rng(1); n = 400
b = rng.uniform(1.0, 1.7, n); i0 = rng.uniform(10, 100, n); t90 = np.log(9 * (360000 - i0) / i0) / b
check = []
for k in range(50):
    d = correr("03_Gusano_CodeRed_logistico", {"Tasa de contacto": round(b[k], 4), "Infectados iniciales": round(i0[k], 2)}); check.append(d.index[(d["Infectados"] >= 0.9 * 360000).argmax()])
plt.hist(t90, bins=np.arange(4, 13.5, 0.5), color=COL["indigo"], alpha=0.85); [plt.axvline(np.percentile(t90, q), ls="--", color="k") for q in (5, 50, 95)]
plt.title("Horas hasta infectar el 90 % (400 corridas)"); plt.xlabel("horas"); plt.ylabel("corridas"); plt.show()
print("P5 = %.2f h · mediana = %.2f h · P95 = %.2f h" % tuple(np.percentile(t90, [5, 50, 95])))
print("Diferencia media analítica vs .mdl (50 corridas, paso 0,0625 h):", round(float(np.mean(np.abs(np.array(check) - t90[:50]))), 3), "h")

---
## 11. Plantilla de Excel
La plantilla tiene una hoja por caso con **los mismos nombres de variables que Vensim** en los encabezados, celdas amarillas para los parámetros y el método de Euler escrito con fórmulas. También trae una hoja de escenarios, un registro de escenarios para llenar y un tablero.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import LineChart, ScatterChart, Reference, Series
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter
# ---------- estilo del libro de Excel ----------
F_TIT = Font(name="Arial", size=16, bold=True, color="0B2545"); F_SUB = Font(name="Arial", size=10, italic=True, color="5B6B7F")
F_HEAD = Font(name="Arial", size=10, bold=True, color="FFFFFF"); FILL_HEAD = PatternFill("solid", fgColor="13315C")
F_IN = Font(name="Arial", size=10, color="0000FF"); FILL_IN = PatternFill("solid", fgColor="FFFF00")
F_BODY = Font(name="Arial", size=10); F_BOLD = Font(name="Arial", size=10, bold=True)
F_KPI = Font(name="Arial", size=16, bold=True, color="0E7C7B"); FILL_KPI = PatternFill("solid", fgColor="EEF4F8")
THIN = Border(*(Side(style="thin", color="D5DEE8"),) * 4)

wb = Workbook(); wb.remove(wb.active)

def hoja(nombre, titulo, subtitulo):
    ws = wb.create_sheet(nombre); ws["A1"] = titulo; ws["A1"].font = F_TIT; ws["A2"] = subtitulo; ws["A2"].font = F_SUB
    ws.sheet_view.showGridLines = False; ws.column_dimensions["A"].width = 34; ws.column_dimensions["B"].width = 14
    return ws

def parametros(ws, fila, filas, titulo="Parámetros (edite las celdas amarillas)"):
    """filas: lista de (etiqueta, valor, nota). Devuelve {etiqueta: 'B#'} con referencias absolutas."""
    ws.cell(fila, 1, titulo).font = F_BOLD; ref = {}
    for i, (lab, val, nota) in enumerate(filas, start=1):
        r = fila + i; ws.cell(r, 1, lab).font = F_BODY; c = ws.cell(r, 2, val); c.font = F_IN; c.fill = FILL_IN; c.border = THIN
        if nota: ws.cell(r, 3, nota).font = F_SUB
        ref[lab] = f"$B${r}"
    return ref

def kpis(ws, fila, col, items, titulo="Indicadores"):
    ws.cell(fila, col, titulo).font = F_BOLD
    for i, (lab, formula, fmt) in enumerate(items, start=1):
        r = fila + i; a = ws.cell(r, col, lab); a.font = F_BODY; a.fill = FILL_KPI; b = ws.cell(r, col + 1, formula); b.font = F_KPI; b.fill = FILL_KPI; b.number_format = fmt
    ws.column_dimensions[get_column_letter(col)].width = 34; ws.column_dimensions[get_column_letter(col + 1)].width = 16

def encabezado(ws, fila, col, nombres, ancho=13):
    for j, n in enumerate(nombres):
        c = ws.cell(fila, col + j, n); c.font = F_HEAD; c.fill = FILL_HEAD; c.alignment = Alignment(horizontal="center", wrap_text=True)
        if ws.column_dimensions[get_column_letter(col + j)].width in (None, 13) or col + j > 2: ws.column_dimensions[get_column_letter(col + j)].width = ancho
    ws.row_dimensions[fila].height = 30

def grafica(ws, ancla, titulo, xcol, ycols, fila1, fila2, xt, yt, hoja_datos=None, ancho=17, alto=8.5, scatter=True, nombres=None):
    src = hoja_datos or ws; ch = ScatterChart() if scatter else LineChart(); ch.title = titulo; ch.style = 13; ch.x_axis.title = xt; ch.y_axis.title = yt; ch.width = ancho; ch.height = alto
    ch.x_axis.delete = False; ch.y_axis.delete = False; ch.legend.position = "b"
    xref = Reference(src, min_col=xcol, min_row=fila1, max_row=fila2); colores = ["0E7C7B", "D1495B", "F4A259", "3E7CB1", "7A6C8F", "0B2545"]
    for i, yc in enumerate(ycols):
        yref = Reference(src, min_col=yc, min_row=fila1 - 1, max_row=fila2)
        s = Series(yref, xref, title_from_data=True); s.marker.symbol = "none"; s.smooth = False
        s.graphicalProperties.line.solidFill = colores[i % 6]; s.graphicalProperties.line.width = 22000; ch.series.append(s)
    ws.add_chart(ch, ancla); return ch

def formato(ws, rango, fmt):
    for row in ws[rango]:
        for c in row: c.number_format = fmt; c.font = F_BODY

F_TIT = Font(name="Arial", size=16, bold=True, color="0F172A"); FILL_HEAD = PatternFill("solid", fgColor="1E293B")
F_KPI = Font(name="Arial", size=16, bold=True, color="0891B2"); FILL_KPI = PatternFill("solid", fgColor="F1F5F9")

def tabla_euler(ws, f0, n, col0, columnas):
    """columnas: [(encabezado, fórmula de la primera fila, fórmula de las demás)]. {r} = fila actual, {p} = fila anterior."""
    encabezado(ws, f0 - 1, col0, [c[0] for c in columnas], 14)
    for i in range(n):
        r = f0 + i
        for j, (_, primera, resto) in enumerate(columnas):
            plantilla = primera if (i == 0 or resto is None) else resto
            ws.cell(r, col0 + j, plantilla.format(r=r, p=r - 1) if isinstance(plantilla, str) else plantilla)
    return f0 + n - 1
wb = Workbook(); wb.remove(wb.active)
print("Funciones de la plantilla listas")

In [ ]:
# --- Hoja genérica de primer orden ---
ws = hoja("Primer orden", "Plantilla · modelo de primer orden", "τ·dx/dt + x = x*. Euler frente a la solución exacta. Pruebe DT = 5, 10 y 15 con τ = 10.")
R = parametros(ws, 4, [("Valor inicial x0", 0, None), ("Equilibrio x*", 1, None), ("Constante de tiempo τ", 10, None), ("Paso DT", 1, "Regla: DT ≤ τ/4")])
x0, xs, ta, dt = R["Valor inicial x0"], R["Equilibrio x*"], R["Constante de tiempo τ"], R["Paso DT"]
fn = tabla_euler(ws, 11, 61, 1, [("t", 0, f"=A{{p}}+{dt}"), ("Euler", f"={x0}", f"=B{{p}}+{dt}*(({xs}-B{{p}})/{ta})"), ("Exacta", f"={xs}+({x0}-{xs})*EXP(-A{{r}}/{ta})", None), ("Error", "=B{r}-C{r}", None)])
formato(ws, f"A11:D{fn}", "0.000")
kpis(ws, 4, 6, [("DT / τ", f"={dt}/{ta}", "0.00"), ("Tiempo para el 95 % (3τ)", f"=3*{ta}", "0.0"), ("Error máximo de Euler", f"=MAX(MAX(D11:D{fn}),-MIN(D11:D{fn}))", "0.0000"), ("Diagnóstico", f'=IF({dt}/{ta}>=1,"DT demasiado grande: inestable",IF({dt}/{ta}>0.25,"DT grande: revise","DT adecuado"))', "@")])
grafica(ws, "I3", "Euler frente a la solución exacta", 1, [2, 3], 11, fn, "t", "x")

# --- Hoja genérica de segundo orden ---
ws = hoja("Segundo orden", "Plantilla · modelo de segundo orden", "x'' + 2ζωn·x' + ωn²·x = ωn²·u escrito como dos niveles: velocidad y posición (Euler semi-implícito).")
R = parametros(ws, 4, [("ζ (amortiguamiento)", 0.3, "Pruebe 0,1 · 0,7 · 1 · 2"), ("ωn (frecuencia natural)", 1, None), ("Entrada u", 1, None), ("Paso DT", 0.05, None)])
z, wn, u, dt = R["ζ (amortiguamiento)"], R["ωn (frecuencia natural)"], R["Entrada u"], R["Paso DT"]
fn = tabla_euler(ws, 11, 401, 1, [("t", 0, f"=A{{p}}+{dt}"), ("Velocidad x1", 0, f"=B{{p}}+{dt}*({wn}^2*({u}-C{{p}})-2*{z}*{wn}*B{{p}})"), ("Posición x2", 0, f"=C{{p}}+{dt}*B{{r}}")])
formato(ws, f"A11:C{fn}", "0.0000")
kpis(ws, 4, 5, [("Sobrepaso teórico Mp", f'=IF({z}<1,EXP(-PI()*{z}/SQRT(1-{z}^2)),0)', "0.0%"), ("Sobrepaso simulado", f"=MAX(0,(MAX(C11:C{fn})-{u})/{u})", "0.0%"), ("Tiempo del pico tp", f'=IF({z}<1,PI()/({wn}*SQRT(1-{z}^2)),"no hay pico")', "0.00"),
    ("Establecimiento ts (2 %)", f"=4/({z}*{wn})", "0.00"), ("Parte real de los polos", f"=-{z}*{wn}", "0.000"), ("Parte imaginaria", f'=IF({z}<1,{wn}*SQRT(1-{z}^2),0)', "0.000"), ("Tipo", f'=IF({z}<0,"Inestable",IF({z}=0,"Oscilación sostenida",IF({z}<1,"Subamortiguado",IF({z}=1,"Crítico","Sobreamortiguado"))))', "@")])
grafica(ws, "H12", "Respuesta al escalón", 1, [3], 11, fn, "t", "x")
print("Hojas genéricas listas")

In [ ]:
# --- CPU ---
ws = hoja("CPU", "Caso 1 · Temperatura de la CPU (01_CPU_primer_orden.mdl)", "Temperatura CPU = INTEG(Calentamiento − Enfriamiento). Paso: 0,5 s.")
R = parametros(ws, 4, [("Temperatura ambiente (°C)", 25, None), ("Resistencia termica (°C/W)", 0.4, "0,75 = disipador sucio"), ("Capacidad termica (J/°C)", 50, None), ("Potencia en reposo (W)", 15, None), ("Potencia en carga (W)", 95, None), ("Inicio de la carga (s)", 30, None), ("Umbral de throttling (°C)", 90, None), ("TIME STEP (s)", 0.5, None)])
Ta, Rt, Ct, P0, P1, ti, um, dt = (R[k] for k in R)
fn = tabla_euler(ws, 15, 401, 1, [("Time", 0, f"=A{{p}}+{dt}"), ("Potencia", f"={P0}+IF(A{{r}}>={ti},{P1}-{P0},0)", None), ("Temperatura CPU", f"={Ta}+{P0}*{Rt}", f"=C{{p}}+{dt}*(D{{p}}-E{{p}})"), ("Calentamiento", f"=B{{r}}/{Ct}", None), ("Enfriamiento", f"=(C{{r}}-{Ta})/({Rt}*{Ct})", None)])
formato(ws, f"A15:E{fn}", "0.00")
kpis(ws, 4, 7, [("Constante de tiempo τ = R·C (s)", f"={Rt}*{Ct}", "0.0"), ("Temperatura de equilibrio en carga", f"={Ta}+{P1}*{Rt}", "0.0"), ("Temperatura final simulada", f"=C{fn}", "0.0"), ("Segundos por encima del umbral", f'=COUNTIF(C15:C{fn},">"&{um})*{dt}', "0.0")])
grafica(ws, "J3", "Temperatura CPU (°C)", 1, [3], 15, fn, "s", "°C")

# --- Tickets ---
ws = hoja("Tickets", "Caso 2 · Cola de tickets (02_Cola_tickets_primer_orden.mdl)", "Tickets pendientes = INTEG(Llegadas − Resoluciones). Ley de Little: B* = λ·τ. Paso: 0,0625 h.")
R = parametros(ws, 4, [("Llegadas normales (tickets/h)", 40, None), ("Llegadas en incidente (tickets/h)", 70, "55 optimista · 100 pesimista"), ("Hora del incidente", 8, None), ("Tiempo de resolucion (h)", 2, "1,4 con más agentes"), ("TIME STEP (h)", 0.0625, None)])
l0, l1, hi, tr, dt = (R[k] for k in R)
fn = tabla_euler(ws, 12, 385, 1, [("Time", 0, f"=A{{p}}+{dt}"), ("Llegadas", f"={l0}+IF(A{{r}}>={hi},{l1}-{l0},0)", None), ("Tickets pendientes", f"={l0}*{tr}", f"=C{{p}}+{dt}*(B{{p}}-D{{p}})"), ("Resoluciones", f"=C{{r}}/{tr}", None), ("Tiempo de espera estimado", "=C{r}/B{r}", None)])
formato(ws, f"A12:E{fn}", "0.00")
kpis(ws, 4, 7, [("Equilibrio antes del incidente", f"={l0}*{tr}", "0.0"), ("Equilibrio después (λ·τ)", f"={l1}*{tr}", "0.0"), ("Tickets al final del día", f"=C{fn}", "0.0"), ("Espera al final (h)", f"=E{fn}", "0.00")])
grafica(ws, "J3", "Tickets pendientes", 1, [3], 12, fn, "hora", "tickets")

# --- Gusano ---
ws = hoja("Gusano", "Caso 3 · Gusano Code-Red (03_Gusano_CodeRed_logistico.mdl)", "Infecciones = Tasa de contacto·S·I/N. Datos: > 359 000 hosts en < 14 h (CAIDA, 2001). Paso: 0,0625 h.")
R = parametros(ws, 4, [("Hosts vulnerables N", 360000, None), ("Infectados iniciales", 30, None), ("Tasa de contacto (1/h)", 1.33, None), ("Tasa de parcheo (1/h)", 0, "Pruebe 0,5"), ("Inicio del parcheo (h)", 4, None), ("TIME STEP (h)", 0.0625, None)])
N_, I0, b_, g_, tp, dt = (R[k] for k in R)
fn = tabla_euler(ws, 13, 257, 1, [("Time", 0, f"=A{{p}}+{dt}"), ("Susceptibles", f"={N_}-{I0}", f"=B{{p}}+{dt}*(-E{{p}}-F{{p}})"), ("Infectados", f"={I0}", f"=C{{p}}+{dt}*(E{{p}}-G{{p}})"), ("Protegidos", 0, f"=D{{p}}+{dt}*(F{{p}}+G{{p}})"),
    ("Infecciones", f"={b_}*B{{r}}*C{{r}}/{N_}", None), ("Parcheo de susceptibles", f"={g_}*B{{r}}*IF(A{{r}}>={tp},1,0)", None), ("Parcheo de infectados", f"={g_}*C{{r}}*IF(A{{r}}>={tp},1,0)", None)])
formato(ws, f"A13:G{fn}", "#,##0.0")
kpis(ws, 4, 9, [("Máximo de infectados", f"=MAX(C13:C{fn})", "#,##0"), ("Hora del pico de infecciones", f"=INDEX(A13:A{fn},MATCH(MAX(E13:E{fn}),E13:E{fn},0))", "0.00"), ("Infecciones por minuto en el pico", f"=MAX(E13:E{fn})/60", "#,##0"), ("Infectados a las 14 h", f"=INDEX(C13:C{fn},ROUND(14/{dt},0)+1)", "#,##0")])
grafica(ws, "L3", "Infectados", 1, [3, 4], 13, fn, "hora", "hosts")

# --- Autoescalado ---
ws = hoja("Autoescalado", "Caso 4 · Autoescalado en la nube (04_Autoescalado_segundo_orden.mdl)", "Regla HPA: deseadas = activas · utilización medida / objetivo. Paso: 0,0625 min.")
R = parametros(ws, 4, [("Demanda base (req/s)", 600, None), ("Demanda pico (req/s)", 1500, None), ("Inicio del pico (min)", 10, None), ("Duracion del pico (min)", 60, None), ("Capacidad por instancia (req/s)", 100, None), ("Utilizacion objetivo", 0.7, None),
    ("Tiempo de ajuste (min)", 1, None), ("Tiempo de arranque (min)", 4, None), ("Tiempo de medicion (min)", 2, None), ("Considerar instancias en arranque", 0, "0 = ingenua · 1 = cuenta las que vienen"), ("TIME STEP (min)", 0.0625, None)])
db, dp, ip, du, cp, uo, ta, tb, tm, cc, dt = (R[k] for k in R)
fn = tabla_euler(ws, 18, 1921, 1, [("Time", 0, f"=A{{p}}+{dt}"), ("Demanda", f"={db}+IF(AND(A{{r}}>={ip},A{{r}}<{ip}+{du}),{dp}-{db},0)", None), ("Instancias en arranque", 0, f"=C{{p}}+{dt}*(I{{p}}-K{{p}})"),
    ("Instancias activas", f"={db}/({cp}*{uo})", f"=D{{p}}+{dt}*(K{{p}}-J{{p}})"), ("Utilizacion medida", f"={uo}", f"=E{{p}}+{dt}*(F{{p}}-E{{p}})/{tm}"), ("Utilizacion real", f"=B{{r}}/({cp}*D{{r}})", None),
    ("Instancias deseadas", f"=D{{r}}*E{{r}}/{uo}", None), ("Brecha de instancias", f"=G{{r}}-D{{r}}-{cc}*C{{r}}", None), ("Solicitud de instancias", f"=MAX(H{{r}}/{ta},0)", None), ("Retiro de instancias", f"=MAX(-H{{r}}/{ta},0)", None), ("Instancias listas", f"=C{{r}}/{tb}", None)])
formato(ws, f"A18:K{fn}", "0.00")
kpis(ws, 4, 13, [("Máximo de instancias activas", f"=MAX(D18:D{fn})", "0.0"), ("Instancias necesarias en el pico", f"={dp}/({cp}*{uo})", "0.0"), ("Sobrepaso", f"=(MAX(D18:D{fn})-{dp}/({cp}*{uo}))/({dp}/({cp}*{uo}))", "0%"),
    ("Minutos con utilización > 90 %", f'=COUNTIF(F18:F{fn},">0.9")*{dt}', "0.0"), ("Costo (instancia-minuto)", f"=SUM(D18:D{fn})*{dt}", "#,##0")])
grafica(ws, "P11", "Instancias activas", 1, [4, 7], 18, fn, "minutos", "instancias")

# --- Brooks ---
ws = hoja("Brooks", "Caso 5 · Proyecto de software y ley de Brooks (05_Proyecto_Ley_de_Brooks.mdl)", "Tres niveles: tareas pendientes, novatos y expertos. Paso: 0,25 días.")
R = parametros(ws, 4, [("Tamano del proyecto (tareas)", 2000, None), ("Expertos iniciales", 10, None), ("Personas a contratar", 0, "Pruebe 10"), ("Dia de contratacion", 150, "Pruebe 0, 100, 150"), ("Tiempo de asimilacion (días)", 60, None), ("Coeficiente de comunicacion", 0.08, None),
    ("Tiempo de mentoria", 0.5, None), ("Productividad experto", 1, None), ("Productividad novato", 0.2, None), ("TIME STEP (días)", 0.25, None)])
W0, E0, pc, dc, tas, kc, mt, pe, pn, dt = (R[k] for k in R)
fn = tabla_euler(ws, 17, 1601, 1, [("Time", 0, f"=A{{p}}+{dt}"), ("Tareas pendientes", f"={W0}", f"=B{{p}}-{dt}*J{{p}}"), ("Novatos", 0, f"=C{{p}}+{dt}*(E{{p}}-F{{p}})"), ("Expertos", f"={E0}", f"=D{{p}}+{dt}*F{{p}}"),
    ("Contratacion", f"=IF(AND(A{{r}}>={dc},A{{r}}<{dc}+{dt}),{pc}/{dt},0)", None), ("Asimilacion", f"=C{{r}}/{tas}", None), ("Tamano del equipo", "=C{r}+D{r}", None), ("Sobrecarga de comunicacion", f"=MIN(0.9,{kc}*(G{{r}}/10)^2)", None),
    ("Productividad bruta", f"=MAX(0,(D{{r}}-{mt}*C{{r}})*{pe}+C{{r}}*{pn})", None), ("Avance", "=IF(B{r}>0,I{r}*(1-H{r}),0)", None)])
formato(ws, f"A17:J{fn}", "0.00")
kpis(ws, 4, 12, [("Día de terminación", f'=COUNTIF(B17:B{fn},">0")*{dt}', "0.0"), ("Equipo máximo", f"=MAX(G17:G{fn})", "0"), ("Sobrecarga máxima", f"=MAX(H17:H{fn})", "0%"), ("Productividad bruta mínima", f"=MIN(I17:I{fn})", "0.00")])
grafica(ws, "O3", "Tareas pendientes", 1, [2], 17, fn, "días", "tareas")
grafica(ws, "O21", "Novatos y expertos", 1, [3, 4], 17, fn, "días", "personas")

# --- Ventilador ---
ws = hoja("Ventilador", "Caso 6 · Control PI del ventilador (06_Ventilador_PI_segundo_orden.mdl)", "Lazo cerrado de segundo orden: temperatura + error acumulado. Paso: 0,25 s.")
R = parametros(ws, 4, [("Kp", 1.5, "Agresivo 0,3 · Bien 1,5 · Lento 1"), ("Ki", 0.1, "Agresivo 0,15 · Bien 0,1 · Lento 0,01"), ("Temperatura objetivo inicial (°C)", 60, None), ("Nuevo objetivo (°C)", 50, None), ("Momento del cambio (s)", 50, None), ("Temperatura ambiente (°C)", 25, None),
    ("Resistencia termica (°C/W)", 0.4, None), ("Efecto del ventilador (°C/%)", 0.3, None), ("Constante de tiempo (s)", 40, None), ("Potencia base (W)", 120, None), ("Potencia extra (W)", 0, "Perturbación"), ("Momento de la carga (s)", 300, None), ("TIME STEP (s)", 0.25, None)])
kp, ki, T0, T1, tc, Ta, Rt, gf, tau_, Pb, Px, tl, dt = (R[k] for k in R)
fan0 = f"(({Ta}+{Pb}*{Rt}-{T0})/{gf})"
fn = tabla_euler(ws, 20, 2401, 1, [("Time", 0, f"=A{{p}}+{dt}"), ("Temperatura objetivo", f"={T0}+IF(A{{r}}>={tc},{T1}-{T0},0)", None), ("Potencia", f"={Pb}+IF(A{{r}}>={tl},{Px},0)", None),
    ("Temperatura servidor", f"={T0}", f"=D{{p}}+{dt}*H{{p}}"), ("Error acumulado", f"={fan0}/{ki}", f"=E{{p}}+{dt}*I{{p}}"), ("Error", "=D{r}-B{r}", None), ("Senal del controlador", f"={kp}*F{{r}}+{ki}*E{{r}}", None),
    ("Cambio de temperatura", f"=({Ta}+C{{r}}*{Rt}-{gf}*J{{r}}-D{{r}})/{tau_}", None), ("Acumulacion del error", "=IF(AND(G{r}>=100,F{r}>0),0,IF(AND(G{r}<=0,F{r}<0),0,F{r}))", None), ("Velocidad del ventilador", "=MAX(0,MIN(100,G{r}))", None)])
formato(ws, f"A20:J{fn}", "0.000")
kpis(ws, 4, 12, [("ζ = (1 + K·Kp)/(2·√(τ·K·Ki))", f"=(1+{gf}*{kp})/(2*SQRT({tau_}*{gf}*{ki}))", "0.00"), ("ωn = √(K·Ki/τ)", f"=SQRT({gf}*{ki}/{tau_})", "0.0000"), ("Sobrepaso teórico", f"=IF(M5<1,EXP(-PI()*M5/SQRT(1-M5^2)),0)", "0.0%"),
    ("Sobrepaso simulado", f"=MAX(0,({T1}-MIN(D20:D{fn}))/({T0}-{T1}))", "0.0%"), ("Ventilador máximo", f"=MAX(J20:J{fn})", "0.0"), ("Temperatura final", f"=D{fn}", "0.00")])
grafica(ws, "O11", "Temperatura del servidor", 1, [4, 2], 20, fn, "s", "°C")
print("Hojas de casos listas")

In [ ]:
# --- Escenarios (solución analítica de la cola) y registro ---
ws = hoja("Escenarios", "Análisis de escenarios · cola de tickets", "Solución exacta de primer orden: B(t) = λ1·τ + (λ0·τ − λ1·τ)·e^(−(t − hora)/τ). Cambie las celdas amarillas de cada escenario.")
ws["A4"] = "Parámetro"; ws["A4"].font = F_BOLD
nombres = ["Optimista", "Base", "Pesimista", "Más agentes"]; vals = [[40, 55, 2], [40, 70, 2], [40, 100, 2], [40, 70, 1.4]]
for j, (nm, v) in enumerate(zip(nombres, vals)):
    c = ws.cell(4, 2 + j, nm); c.font = F_HEAD; c.fill = FILL_HEAD
    for i, x in enumerate(v):
        cc_ = ws.cell(5 + i, 2 + j, x); cc_.font = F_IN; cc_.fill = FILL_IN
for i, lab in enumerate(["λ0 antes (tickets/h)", "λ1 en el incidente (tickets/h)", "τ (h)"]): ws.cell(5 + i, 1, lab).font = F_BODY
ws["A8"] = "Hora del incidente"; c = ws["B8"]; c.value = 8; c.font = F_IN; c.fill = FILL_IN
ws["A10"] = "Equilibrio λ1·τ"; ws["A11"] = "Espera W = τ (h)"; ws["A12"] = "Tickets a las 12 h"
for j in range(4):
    L = get_column_letter(2 + j); ws[f"{L}10"] = f"={L}6*{L}7"; ws[f"{L}11"] = f"={L}7"
for j in range(4): ws.cell(10, 2 + j).number_format = "0.0"
encabezado(ws, 14, 1, ["Hora"] + nombres)
for i in range(49):
    r = 15 + i; ws.cell(r, 1, i * 0.5)
    for j in range(4):
        L = get_column_letter(2 + j); ws.cell(r, 2 + j, f"=IF(A{r}<$B$8,{L}$5*{L}$7,{L}$6*{L}$7+({L}$5*{L}$7-{L}$6*{L}$7)*EXP(-(A{r}-$B$8)/{L}$7))").number_format = "0.0"
for j in range(4): L = get_column_letter(2 + j); ws[f"{L}12"] = f"=INDEX({L}15:{L}63,25)"; ws[f"{L}12"].number_format = "0.0"
grafica(ws, "H3", "Tickets pendientes por escenario", 1, [2, 3, 4, 5], 15, 63, "hora", "tickets")
ws.cell(66, 1, "Registro de escenarios (llénelo con sus corridas de Vensim, Python o Excel)").font = F_BOLD
encabezado(ws, 67, 1, ["Escenario", "Modelo", "Parámetro cambiado", "Valor", "Resultado clave", "Conclusión"], 22)
ejemplo = ["Pesimista", "02 Tickets", "Llegadas en incidente", 100, "200 tickets en equilibrio", "Hay que reducir τ, no esperar"]
for j, v in enumerate(ejemplo): ws.cell(68, 1 + j, v).font = F_SUB
for r in range(69, 79):
    for j in range(6): c = ws.cell(r, 1 + j); c.fill = FILL_IN; c.font = F_IN; c.border = THIN
ws.cell(79, 1, "La fila 68 es un ejemplo; escriba sus escenarios en las filas amarillas.").font = F_SUB

# --- Tablero e instrucciones ---
ws = wb.create_sheet("Tablero", 0); ws.sheet_view.showGridLines = False
ws["A1"] = "Tablero · modelos de primer y segundo orden"; ws["A1"].font = F_TIT; ws["A2"] = "Cada cifra es una fórmula que apunta a la hoja del caso."; ws["A2"].font = F_SUB
tiles = [("CPU · temperatura final (°C)", "=CPU!H7", "0.0"), ("CPU · constante de tiempo (s)", "=CPU!H5", "0.0"), ("Tickets · equilibrio tras el incidente", "=Tickets!H6", "0.0"),
    ("Gusano · máximo de infectados", "=Gusano!J5", "#,##0"), ("Gusano · hora del pico", "=Gusano!J6", "0.00"), ("Nube · máximo de instancias", "=Autoescalado!N5", "0.0"),
    ("Nube · sobrepaso", "=Autoescalado!N7", "0%"), ("Brooks · día de terminación", "=Brooks!M5", "0.0"), ("Ventilador · ζ", "=Ventilador!M5", "0.00"),
    ("Ventilador · sobrepaso simulado", "=Ventilador!M8", "0.0%"), ("2.º orden · sobrepaso teórico", "='Segundo orden'!F5", "0.0%"), ("1.er orden · diagnóstico de DT", "='Primer orden'!G8", "@")]
for i, (lab, f, fmt) in enumerate(tiles):
    r = 4 + (i // 3) * 3; c = 1 + (i % 3) * 3
    a = ws.cell(r, c, lab); a.font = Font(name="Arial", size=9, color="64748B"); a.fill = FILL_KPI
    b = ws.cell(r + 1, c, f); b.font = Font(name="Arial", size=18, bold=True, color="0891B2"); b.fill = FILL_KPI; b.number_format = fmt
    ws.merge_cells(start_row=r, start_column=c, end_row=r, end_column=c + 1); ws.merge_cells(start_row=r + 1, start_column=c, end_row=r + 1, end_column=c + 1)
for col in range(1, 10): ws.column_dimensions[get_column_letter(col)].width = 20 if col % 3 else 3
grafica(ws, "A17", "Autoescalado: instancias activas", 1, [4], 18, 18 + 1920, "minutos", "instancias", hoja_datos=wb["Autoescalado"], ancho=16, alto=7.5)
grafica(ws, "F17", "Brooks: tareas pendientes", 1, [2], 17, 17 + 1600, "días", "tareas", hoja_datos=wb["Brooks"], ancho=16, alto=7.5)
grafica(ws, "A33", "Ventilador: temperatura", 1, [4], 20, 20 + 2400, "s", "°C", hoja_datos=wb["Ventilador"], ancho=16, alto=7.5)
grafica(ws, "F33", "Gusano: infectados", 1, [3], 13, 13 + 256, "hora", "hosts", hoja_datos=wb["Gusano"], ancho=16, alto=7.5)

ws = wb.create_sheet("Instrucciones", 0); ws.sheet_view.showGridLines = False; ws.column_dimensions["A"].width = 24; ws.column_dimensions["B"].width = 100
ws["A1"] = "Plantilla · Modelos de primer y segundo orden"; ws["A1"].font = F_TIT
ws["A2"] = "Unidad Modular III. Cada hoja de caso reproduce un modelo de Vensim con los mismos nombres de variables."; ws["A2"].font = F_SUB
c = ws["A4"]; c.value = "Parámetro"; c.font = F_IN; c.fill = FILL_IN; ws["B4"] = "Celdas amarillas con texto azul: cámbielas para crear escenarios."
c = ws["A5"]; c.value = "Indicador"; c.font = F_KPI; c.fill = FILL_KPI; ws["B5"] = "Resultados calculados con fórmulas."
ws["A6"] = "Tabla"; ws["B6"] = "Cada fila es un paso de Euler: nivel(t + DT) = nivel(t) + DT · (entradas − salidas). Es lo mismo que hace Vensim con TIME STEP."
ws["A7"] = "Nombres"; ws["B7"] = "Los encabezados de las columnas son los nombres de las variables en los archivos .mdl (sin tildes, como en Vensim)."
ws["A9"] = "Hojas"; ws["A9"].font = F_BOLD
indice = [("Tablero", "Indicadores de todos los casos."), ("Primer orden", "Plantilla genérica: Euler frente a la solución exacta y efecto del paso DT."), ("Segundo orden", "Plantilla genérica: ζ, ωn, sobrepaso, tiempos y polos."),
    ("CPU", "Caso 1 · 01_CPU_primer_orden.mdl"), ("Tickets", "Caso 2 · 02_Cola_tickets_primer_orden.mdl"), ("Gusano", "Caso 3 · 03_Gusano_CodeRed_logistico.mdl"), ("Autoescalado", "Caso 4 · 04_Autoescalado_segundo_orden.mdl"),
    ("Brooks", "Caso 5 · 05_Proyecto_Ley_de_Brooks.mdl"), ("Ventilador", "Caso 6 · 06_Ventilador_PI_segundo_orden.mdl"), ("Escenarios", "Cuatro escenarios de la cola con solución exacta y registro de escenarios para llenar.")]
for i, (h, d) in enumerate(indice):
    r = 10 + i; c = ws.cell(r, 1, h); c.hyperlink = f"#'{h}'!A1"; c.font = Font(name="Arial", size=10, color="0891B2", underline="single"); ws.cell(r, 2, d).font = F_BODY
for row in ws["A4:B21"]:
    for c in row:
        if c.value and c.font == Font(): c.font = F_BODY
wb.calculation.fullCalcOnLoad = True
ARCHIVO = "Plantilla_Modelos_Primer_Segundo_Orden.xlsx"; wb.save(ARCHIVO); print("Guardado:", ARCHIVO, "· hojas:", wb.sheetnames)

In [ ]:
# Descargas (en Colab)
with zipfile.ZipFile("Modelos_Vensim.zip", "w") as z:
    for f in sorted(os.listdir("vensim")): z.write(f"vensim/{f}", f)
try:
    from google.colab import files
    files.download(ARCHIVO); files.download("Modelos_Vensim.zip")
except ImportError:
    print("Fuera de Colab: los archivos quedaron en la carpeta de trabajo.")

---
## 12. Ejercicios
1. Con `correr("01_CPU_primer_orden", {...})`, encuentre la resistencia térmica máxima que mantiene la CPU bajo 85 °C con 110 W y 30 °C de ambiente. Compruebe con $T^* = T_a + P R_{th}$.
2. Simule el modelo de tickets con `time_step=2.5` en el intérprete mínimo. Explique el resultado con $DT/\tau$.
3. Calcule con la fórmula la hora del pico del gusano y compárela con la simulación.
4. Defina un escenario pesimista para el autoescalado (demanda pico 2 000 y arranque de 6 min) y repórtelo con los tres indicadores del caso 4.
5. Haga el tornado del día de terminación del proyecto (modelo 05) para cinco parámetros.
6. Diseñe un PI con ζ = 0,7 y ωn = 0,04 rad/s y verifíquelo en Python, en Vensim y en la hoja `Ventilador` de la plantilla.
7. Construya en Vensim un modelo de memoria con fuga (memory leak) y tradúzcalo a Python con `simular_mdl`.